In [1]:
from transformers import pipeline
from transformers import AutoModel, AutoTokenizer
import itertools
import torch

# load model
model = AutoModel.from_pretrained("aneuraz/awesome-align-with-co")
tokenizer = AutoTokenizer.from_pretrained("aneuraz/awesome-align-with-co")

# model parameters
align_layer = 8
threshold = 1e-3

device = 0 if torch.cuda.is_available() else -1
translator = pipeline(
    "translation_de_to_en",
    model="Helsinki-NLP/opus-mt-de-en",
    device=device
)
model.to("cuda" if torch.cuda.is_available() else "cpu")


Some weights of BertModel were not initialized from the model checkpoint at aneuraz/awesome-align-with-co and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

In [3]:
import ipywidgets as widgets
from IPython.display import display
#1️⃣ Create a text input area and a button using ipywidgets.
text_area = widgets.Textarea(
    placeholder='Paste German text here...',
    layout=widgets.Layout(width='100%', height='150px')
)
process_button = widgets.Button(description='Process')

#2️⃣ Display them in the Jupyter notebook.
display(text_area, process_button)

#3️⃣ Attach a callback function to the button that stores the textarea content in a global Python
german_text = ""
def on_process_clicked(b):
    global german_text
    german_text = text_area.value
    print("✔ Text received:", len(german_text), "characters")

process_button.on_click(on_process_clicked)

Textarea(value='', layout=Layout(height='150px', width='100%'), placeholder='Paste German text here...')

Button(description='Process', style=ButtonStyle())

In [5]:
print(german_text)

Die Wissenschaftler stellten Hindernisse in Form greller Lichter auf, denn Physarum mag kein Licht.


In [6]:
import spacy

nlp = spacy.load("de_core_news_sm")
#Split the original German text into sentences using spaCy.
german_sents = list(nlp(german_text).sents)

In [7]:
def tokenAlignment(src):
    # Translate DE → EN
    tgt = translator(src)[0]["translation_text"]

    #print("SOURCE (DE):", src)
    #print("TARGET (EN):", tgt)
    #print("-" * 100)

    # pre-processing
    sent_src, sent_tgt = src.strip().split(), tgt.strip().split()
    token_src, token_tgt = [tokenizer.tokenize(word) for word in sent_src], [tokenizer.tokenize(word) for word in sent_tgt]
    wid_src, wid_tgt = [tokenizer.convert_tokens_to_ids(x) for x in token_src], [tokenizer.convert_tokens_to_ids(x) for x in token_tgt]
    ids_src, ids_tgt = tokenizer.prepare_for_model(list(itertools.chain(*wid_src)), return_tensors='pt', model_max_length=tokenizer.model_max_length, truncation=True)['input_ids'], tokenizer.prepare_for_model(list(itertools.chain(*wid_tgt)), return_tensors='pt', truncation=True, model_max_length=tokenizer.model_max_length)['input_ids']
    sub2word_map_src = []
    for i, word_list in enumerate(token_src):
        sub2word_map_src += [i for x in word_list]
    sub2word_map_tgt = []
    for i, word_list in enumerate(token_tgt):
        sub2word_map_tgt += [i for x in word_list]

    # alignment
    align_layer = 8
    threshold = 1e-3
    model.eval()
    with torch.no_grad():
      out_src = model(ids_src.unsqueeze(0), output_hidden_states=True)[2][align_layer][0, 1:-1]
      out_tgt = model(ids_tgt.unsqueeze(0), output_hidden_states=True)[2][align_layer][0, 1:-1]

      dot_prod = torch.matmul(out_src, out_tgt.transpose(-1, -2))
      #print("-"*100)
      msg = "Remember we are working with subwords here"
      #print(msg,"-"*(99-len(msg)))
      #print(dot_prod)
      #print("-"*100)
      softmax_srctgt = torch.nn.Softmax(dim=-1)(dot_prod)
      softmax_tgtsrc = torch.nn.Softmax(dim=-2)(dot_prod)

      softmax_inter = (softmax_srctgt > threshold)*(softmax_tgtsrc > threshold)

    align_subwords = torch.nonzero(softmax_inter, as_tuple=False)
    align_words = set()
    for i, j in align_subwords:
        align_words.add( (sub2word_map_src[i], sub2word_map_tgt[j]) )

    # printing
    class color:
       PURPLE = '\033[95m'
       CYAN = '\033[96m'
       DARKCYAN = '\033[36m'
       BLUE = '\033[94m'
       GREEN = '\033[92m'
       YELLOW = '\033[93m'
       RED = '\033[91m'
       BOLD = '\033[1m'
       UNDERLINE = '\033[4m'
       END = '\033[0m'

    for i, j in sorted(align_words):
        print(f'{color.BOLD}{color.BLUE}{sent_src[i]}{color.END}==={color.BOLD}{color.RED}{sent_tgt[j]}{color.END}')

    aligned_pairs = []
    for i, j in sorted(align_words):
        aligned_pairs.append({
            "src_word": sent_src[i],
            "tgt_word": sent_tgt[j],
            "src_idx": i,
            "tgt_idx": j
        })

    return aligned_pairs


In [8]:
def alignment_grid(aligned_pairs):
    rows = []

    row_layout = widgets.Layout(
        display='flex',
        flex_flow='row nowrap',
        align_items='center'
    )

    col_keep = widgets.Layout(flex='0 0 60px')
    col_word = widgets.Layout(flex='0 0 150px')

    # Header
    header = widgets.HBox([
        widgets.HTML("<b>Keep</b>", layout=col_keep),
        widgets.HTML("<b>German</b>", layout=col_word),
        widgets.HTML("<b>English</b>", layout=col_word),
    ], layout=row_layout)
    rows.append(header)

    # Rows
    for pair in aligned_pairs:
        cb = widgets.Checkbox(value=True, layout=col_keep)
        cb.pair = pair

        row = widgets.HBox([
            cb,
            widgets.Label(pair["src_word"], layout=col_word),
            widgets.Label(pair["tgt_word"], layout=col_word),
        ], layout=row_layout)

        rows.append(row)

    container = widgets.VBox(rows)

    def get_selected():
        return [
            row.children[0].pair
            for row in container.children[1:]
            if row.children[0].value
        ]

    return container, get_selected


In [26]:
all_getters = []
all_kept = []
for sent in german_sents:
    aligned_pairs = tokenAlignment(sent.text)
    grid, get_selected = alignment_grid(aligned_pairs)

    display(grid)
    all_getters.append(get_selected)

save_btn = widgets.Button(description="Save all words")

def on_save_clicked(b):
    print("fuck off")
    for getter in all_getters:
        all_kept.extend(getter())

    print(f"Saved {len(all_kept)} total words")
    print(all_kept)

save_btn.on_click(on_save_clicked)
display(save_btn)


Die===The
Wissenschaftler===scientists
stellten===set
Hindernisse===obstacles
in===in
Form===form
greller===bright
Lichter===lights,
auf,===up
auf,===lights,
denn===because
Physarum===Physarum
mag===does
kein===not
Licht.===light.


Button(description='Save all words', style=ButtonStyle())

In [27]:
print(get_selected)

<function alignment_grid.<locals>.get_selected at 0x000001B40AD916C0>


In [28]:
print(aligned_pairs)

[{'src_word': 'Die', 'tgt_word': 'The', 'src_idx': 0, 'tgt_idx': 0}, {'src_word': 'Wissenschaftler', 'tgt_word': 'scientists', 'src_idx': 1, 'tgt_idx': 1}, {'src_word': 'stellten', 'tgt_word': 'set', 'src_idx': 2, 'tgt_idx': 2}, {'src_word': 'Hindernisse', 'tgt_word': 'obstacles', 'src_idx': 3, 'tgt_idx': 4}, {'src_word': 'in', 'tgt_word': 'in', 'src_idx': 4, 'tgt_idx': 5}, {'src_word': 'Form', 'tgt_word': 'form', 'src_idx': 5, 'tgt_idx': 7}, {'src_word': 'greller', 'tgt_word': 'bright', 'src_idx': 6, 'tgt_idx': 9}, {'src_word': 'Lichter', 'tgt_word': 'lights,', 'src_idx': 7, 'tgt_idx': 10}, {'src_word': 'auf,', 'tgt_word': 'up', 'src_idx': 8, 'tgt_idx': 3}, {'src_word': 'auf,', 'tgt_word': 'lights,', 'src_idx': 8, 'tgt_idx': 10}, {'src_word': 'denn', 'tgt_word': 'because', 'src_idx': 9, 'tgt_idx': 11}, {'src_word': 'Physarum', 'tgt_word': 'Physarum', 'src_idx': 10, 'tgt_idx': 12}, {'src_word': 'mag', 'tgt_word': 'does', 'src_idx': 11, 'tgt_idx': 13}, {'src_word': 'kein', 'tgt_word': '

In [29]:
print(grid)

In [34]:
print(all_kept)

[{'src_word': 'Die', 'tgt_word': 'The', 'src_idx': 0, 'tgt_idx': 0}, {'src_word': 'Wissenschaftler', 'tgt_word': 'scientists', 'src_idx': 1, 'tgt_idx': 1}, {'src_word': 'stellten', 'tgt_word': 'set', 'src_idx': 2, 'tgt_idx': 2}, {'src_word': 'Hindernisse', 'tgt_word': 'obstacles', 'src_idx': 3, 'tgt_idx': 4}, {'src_word': 'in', 'tgt_word': 'in', 'src_idx': 4, 'tgt_idx': 5}, {'src_word': 'Form', 'tgt_word': 'form', 'src_idx': 5, 'tgt_idx': 7}, {'src_word': 'greller', 'tgt_word': 'bright', 'src_idx': 6, 'tgt_idx': 9}, {'src_word': 'Lichter', 'tgt_word': 'lights,', 'src_idx': 7, 'tgt_idx': 10}, {'src_word': 'auf,', 'tgt_word': 'up', 'src_idx': 8, 'tgt_idx': 3}, {'src_word': 'auf,', 'tgt_word': 'lights,', 'src_idx': 8, 'tgt_idx': 10}, {'src_word': 'denn', 'tgt_word': 'because', 'src_idx': 9, 'tgt_idx': 11}, {'src_word': 'Physarum', 'tgt_word': 'Physarum', 'src_idx': 10, 'tgt_idx': 12}, {'src_word': 'Die', 'tgt_word': 'The', 'src_idx': 0, 'tgt_idx': 0}, {'src_word': 'Wissenschaftler', 'tgt_

In [31]:
print(all_getters)

[<function alignment_grid.<locals>.get_selected at 0x000001B40AD916C0>]
